In [2]:
# ============================================================
# Housing Affordability DSS — Notebook 01: EDA
# CS7P01: MSc Project | London Metropolitan University
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# ── Plot styling ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize':    (13, 6),
    'axes.titlesize':    14,
    'axes.labelsize':    12,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'serif',
    'axes.titlepad':     14
})

PALETTE = [
    '#1F3864', '#2E5090', '#C0392B', '#27AE60', '#E67E22',
    '#8E44AD', '#2980B9', '#D35400', '#16A085', '#2C3E50'
]

# ── Paths ─────────────────────────────────────────────────────
RAW       = '../data/raw/'
PROCESSED = '../data/processed/'
os.makedirs(PROCESSED, exist_ok=True)

# ── Verified region list (exact strings from source file) ───────
REGIONS = [
    'East Midlands',
    'East of England',
    'London',
    'North East',
    'North West',
    'South East',
    'South West',
    'West Midlands Region',   # NOTE: NOT 'West Midlands' — that is the LA
    'Yorkshire and The Humber',
    'Wales'
]

print("✓ Environment ready")
print(f"  Regions defined : {len(REGIONS)}")
print(f"  Raw path        : {os.path.abspath(RAW)}")
print(f"  Processed path  : {os.path.abspath(PROCESSED)}")

✓ Environment ready
  Regions defined : 10
  Raw path        : /Users/sushansunuwar/LondonMet/housing-dss/data/raw
  Processed path  : /Users/sushansunuwar/LondonMet/housing-dss/data/processed


In [3]:
# ============================================================
# DATASET 1: Land Registry UK HPI Full File 2025
# Monthly house price data by region and local authority
# Source: https://www.gov.uk/government/collections/uk-house-price-index
#
# Columns used:
#   Date            — monthly date of record
#   RegionName      — geography (we filter to 10 regions)
#   AveragePrice    — mix-adjusted mean price, all properties
#   AveragePriceSA  — seasonally adjusted average price
#   SalesVolume     — number of transactions (proxy for market activity)
#   12m%Change      — year-on-year % price change (pre-computed)
#   IndexSA         — seasonally adjusted index (base Jan 2015 = 100)
# ============================================================

hpi_raw = pd.read_csv(
    RAW + 'land_registry_hpi_full_2025.csv',
    encoding='utf-8'
)

print(f"── Raw HPI shape : {hpi_raw.shape}")
print(f"── Columns       : {hpi_raw.columns.tolist()}")

── Raw HPI shape : (149085, 54)
── Columns       : ['Date', 'RegionName', 'AreaCode', 'AveragePrice', 'Index', 'IndexSA', '1m%Change', '12m%Change', 'AveragePriceSA', 'SalesVolume', 'DetachedPrice', 'DetachedIndex', 'Detached1m%Change', 'Detached12m%Change', 'SemiDetachedPrice', 'SemiDetachedIndex', 'SemiDetached1m%Change', 'SemiDetached12m%Change', 'TerracedPrice', 'TerracedIndex', 'Terraced1m%Change', 'Terraced12m%Change', 'FlatPrice', 'FlatIndex', 'Flat1m%Change', 'Flat12m%Change', 'CashPrice', 'CashIndex', 'Cash1m%Change', 'Cash12m%Change', 'CashSalesVolume', 'MortgagePrice', 'MortgageIndex', 'Mortgage1m%Change', 'Mortgage12m%Change', 'MortgageSalesVolume', 'FTBPrice', 'FTBIndex', 'FTB1m%Change', 'FTB12m%Change', 'FOOPrice', 'FOOIndex', 'FOO1m%Change', 'FOO12m%Change', 'NewPrice', 'NewIndex', 'New1m%Change', 'New12m%Change', 'NewSalesVolume', 'OldPrice', 'OldIndex', 'Old1m%Change', 'Old12m%Change', 'OldSalesVolume']


In [4]:
# ── Filter to our 10 regions only ────────────────────────────
hpi = hpi_raw[hpi_raw['RegionName'].isin(REGIONS)].copy()

# ── Parse date ───────────────────────────────────────────────
# Format is DD/MM/YYYY — must use dayfirst=True
hpi['Date'] = pd.to_datetime(hpi['Date'], dayfirst=True)
hpi['Year']  = hpi['Date'].dt.year
hpi['Month'] = hpi['Date'].dt.month

# ── Select and rename columns we will use ────────────────────
hpi = hpi[[
    'Date', 'Year', 'Month', 'RegionName',
    'AveragePrice', 'AveragePriceSA',
    'SalesVolume', '12m%Change', 'IndexSA'
]].copy()

hpi.columns = [
    'date', 'year', 'month', 'region',
    'avg_price', 'avg_price_sa',
    'sales_volume', 'yoy_pct_change', 'index_sa'
]

# ── Convert numeric columns — some may have come in as strings
for col in ['avg_price', 'avg_price_sa', 'sales_volume',
            'yoy_pct_change', 'index_sa']:
    hpi[col] = pd.to_numeric(hpi[col], errors='coerce')

# ── Sort ──────────────────────────────────────────────────────
hpi = hpi.sort_values(['region', 'date']).reset_index(drop=True)

# ── Verification ──────────────────────────────────────────────
print(f"\n── Filtered HPI shape : {hpi.shape}")
print(f"── Regions found      : {sorted(hpi['region'].unique())}")
print(f"── Date range         : {hpi['date'].min()} → {hpi['date'].max()}")
print(f"\n── Missing values:\n{hpi.isnull().sum()}")
hpi.head(10)


── Filtered HPI shape : (5745, 9)
── Regions found      : ['East Midlands', 'East of England', 'London', 'North East', 'North West', 'South East', 'South West', 'Wales', 'West Midlands Region', 'Yorkshire and The Humber']
── Date range         : 1968-04-01 00:00:00 → 2025-12-01 00:00:00

── Missing values:
date                 0
year                 0
month                0
region               0
avg_price            0
avg_price_sa      2025
sales_volume      2045
yoy_pct_change     120
index_sa          2025
dtype: int64


,date,year,month,region,avg_price,avg_price_sa,sales_volume,yoy_pct_change,index_sa
0,1968-04-01,1968,4,East Midlands,2914,NaN,NaN,NaN,NaN
1,1968-05-01,1968,5,East Midlands,2914,NaN,NaN,NaN,NaN
2,1968-06-01,1968,6,East Midlands,2914,NaN,NaN,NaN,NaN
3,1968-07-01,1968,7,East Midlands,3057,NaN,NaN,NaN,NaN
4,1968-08-01,1968,8,East Midlands,3057,NaN,NaN,NaN,NaN
5,1968-09-01,1968,9,East Midlands,3057,NaN,NaN,NaN,NaN
6,1968-10-01,1968,10,East Midlands,2961,NaN,NaN,NaN,NaN
7,1968-11-01,1968,11,East Midlands,2961,NaN,NaN,NaN,NaN
8,1968-12-01,1968,12,East Midlands,2961,NaN,NaN,NaN,NaN
9,1969-01-01,1969,1,East Midlands,2995,NaN,NaN,NaN,NaN


In [5]:
# ============================================================
# SANITY CHECK — Verify each region has the expected row count
# Monthly data from ~1995 to 2025 = ~360 rows per region
# Flag any region with significantly fewer rows
# ============================================================

region_counts = hpi.groupby('region').size().reset_index(name='row_count')
region_counts['date_start'] = hpi.groupby('region')['date'].min().values
region_counts['date_end']   = hpi.groupby('region')['date'].max().values

print("── Rows per region:")
print(region_counts.to_string(index=False))

# Check for any region with unexpected gaps
expected_months = (hpi['date'].max().year - hpi['date'].min().year) * 12
print(f"\n── Expected months per region (approx): {expected_months}")
print(f"── Any regions with gaps? "
      f"{region_counts[region_counts['row_count'] < expected_months * 0.95]['region'].tolist()}")

── Rows per region:
                  region  row_count date_start   date_end
           East Midlands        693 1968-04-01 2025-12-01
         East of England        405 1992-04-01 2025-12-01
                  London        693 1968-04-01 2025-12-01
              North East        405 1992-04-01 2025-12-01
              North West        372 1995-01-01 2025-12-01
              South East        405 1992-04-01 2025-12-01
              South West        693 1968-04-01 2025-12-01
                   Wales        693 1968-04-01 2025-12-01
    West Midlands Region        693 1968-04-01 2025-12-01
Yorkshire and The Humber        693 1968-04-01 2025-12-01

── Expected months per region (approx): 684
── Any regions with gaps? ['East of England', 'North East', 'North West', 'South East']


In [6]:
# ============================================================
# CELL 4 — Apply Date Filter: 1995 onwards
#
# Rationale: ONS affordability ratio series begins in 1997.
# We use 1995 as our start to retain 2 years of price data
# before the ratio series begins — useful for context plots
# and lag features in modelling.
# Pre-1995 data has no matching earnings/ratio data and
# contains no seasonally adjusted values — not useful.
# ============================================================

hpi = hpi[hpi['year'] >= 1995].copy().reset_index(drop=True)

# ── Re-check after filter ─────────────────────────────────────
print(f"── Filtered shape (1995+) : {hpi.shape}")
print(f"── Date range             : {hpi['date'].min()} → {hpi['date'].max()}")
print(f"\n── Missing values after filter:")
print(hpi.isnull().sum())

print(f"\n── Rows per region after filter:")
print(hpi.groupby('region').agg(
    rows       = ('date', 'count'),
    date_start = ('date', 'min'),
    date_end   = ('date', 'max')
).to_string())

── Filtered shape (1995+) : (3720, 9)
── Date range             : 1995-01-01 00:00:00 → 2025-12-01 00:00:00

── Missing values after filter:
date               0
year               0
month              0
region             0
avg_price          0
avg_price_sa       0
sales_volume      20
yoy_pct_change    12
index_sa           0
dtype: int64

── Rows per region after filter:
                          rows date_start   date_end
region                                              
East Midlands              372 1995-01-01 2025-12-01
East of England            372 1995-01-01 2025-12-01
London                     372 1995-01-01 2025-12-01
North East                 372 1995-01-01 2025-12-01
North West                 372 1995-01-01 2025-12-01
South East                 372 1995-01-01 2025-12-01
South West                 372 1995-01-01 2025-12-01
Wales                      372 1995-01-01 2025-12-01
West Midlands Region       372 1995-01-01 2025-12-01
Yorkshire and The Humber   372 1995-01-0

In [7]:
# Save cleaned HPI to processed folder
hpi.to_csv(PROCESSED + 'hpi_clean.csv', index=False)
print("✓ hpi_clean.csv saved")

✓ hpi_clean.csv saved


In [9]:
# ============================================================
# CELL 5 — Load ONS HP Earnings Ratio: Sheet 1c
# Median affordability ratio by country and region 1997–2025
# This is your PRIMARY affordability series — the main
# target variable for all modelling and forecasting.
#
# File structure (verified from screenshot):
#   Row 1  = title (skip)
#   Row 2  = headers: Code, Name, 1997, 1998 ... 2025
#   Row 3+ = data in wide format (regions as rows, years as cols)
# ============================================================

ratio_raw = pd.read_excel(
    RAW + 'ons_hp_earnings_ratio.xlsx',
    sheet_name='1c',
    header=1        # Row 2 is the header (0-indexed = 1)
)

print(f"── Raw shape        : {ratio_raw.shape}")
print(f"── Columns          : {ratio_raw.columns.tolist()}")
print(f"\n── First 5 rows:")
ratio_raw.head()

── Raw shape        : (12, 32)
── Columns          : ['Code', 'Name', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '5-Year Average']

── First 5 rows:


,Code,Name,1997,1998,1999,2000,2001,2002,2003,2004,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,5-Year Average
0,K04000001,England and Wales,3.55,3.66,3.89,4.13,4.42,5.06,5.85,6.53,...,7.77,7.85,7.73,7.82,8.95,8.45,8.28,7.74,7.55,8.19
1,E92000001,England,3.54,3.67,3.96,4.19,4.50,5.12,5.93,6.60,...,7.91,8.04,7.88,7.86,9.06,8.56,8.45,7.84,7.63,8.31
2,W92000004,Wales,3.00,3.04,3.16,3.26,3.33,3.71,4.31,5.43,...,5.76,5.92,5.83,5.88,6.57,6.43,6.24,6.13,5.95,6.26
3,E12000001,North East,2.98,3.01,3.07,2.98,3.02,3.29,3.95,4.78,...,5.21,5.31,5.21,5.21,5.81,5.25,5.34,5.07,5.00,5.29
4,E12000002,North West,3.01,3.01,3.09,3.13,3.23,3.51,4.02,4.87,...,5.79,5.84,5.86,5.91,6.78,6.52,6.35,6.11,5.92,6.34


In [10]:
# ============================================================
# CELL 6 — Clean and reshape ratio data
#
# Current shape: wide — each year is a column
# Target shape:  long — one row per region per year
#
# Regions in this file use 'West Midlands' (not 'West Midlands
# Region' as in HPI) — we standardise naming here.
# ============================================================

# ── Define region mapping ─────────────────────────────────────
# Maps ONS sheet names → our standard REGIONS list
# Only keep the 10 regions we are modelling
RATIO_REGION_MAP = {
    'England and Wales':          'England and Wales',  # national benchmark
    'England':                    'England',            # national benchmark
    'Wales':                      'Wales',
    'North East':                 'North East',
    'North West':                 'North West',
    'Yorkshire and The Humber':   'Yorkshire and The Humber',
    'East Midlands':              'East Midlands',
    'West Midlands':              'West Midlands Region',  # standardise name
    'East of England':            'East of England',
    'London':                     'London',
    'South East':                 'South East',
    'South West':                 'South West',
}

# ── Filter to our regions ─────────────────────────────────────
ratio_filtered = ratio_raw[
    ratio_raw['Name'].isin(RATIO_REGION_MAP.keys())
].copy()

# ── Identify year columns ─────────────────────────────────────
year_cols = [c for c in ratio_filtered.columns
             if str(c).strip().isdigit() and 1997 <= int(str(c).strip()) <= 2025]

print(f"── Year columns found : {year_cols}")
print(f"── Regions before melt: {ratio_filtered['Name'].tolist()}")

# ── Melt wide → long ──────────────────────────────────────────
ratio_long = ratio_filtered.melt(
    id_vars=['Code', 'Name'],
    value_vars=year_cols,
    var_name='year',
    value_name='affordability_ratio'
)

# ── Clean up ──────────────────────────────────────────────────
ratio_long['year'] = pd.to_numeric(ratio_long['year'], errors='coerce').astype('Int64')
ratio_long['affordability_ratio'] = pd.to_numeric(ratio_long['affordability_ratio'], errors='coerce')

# Standardise region names to match HPI file
ratio_long['region'] = ratio_long['Name'].map(RATIO_REGION_MAP)

# Drop Code and Name — replaced by standardised region
ratio_long = ratio_long.drop(columns=['Code', 'Name'])
ratio_long = ratio_long.dropna(subset=['affordability_ratio'])
ratio_long = ratio_long.sort_values(['region', 'year']).reset_index(drop=True)

print(f"\n── Long format shape  : {ratio_long.shape}")
print(f"── Regions            : {sorted(ratio_long['region'].unique())}")
print(f"── Year range         : {ratio_long['year'].min()} → {ratio_long['year'].max()}")
print(f"\n── Missing values:\n{ratio_long.isnull().sum()}")
ratio_long.head(12)

── Year columns found : ['1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
── Regions before melt: ['England and Wales', 'England', 'Wales', 'North East', 'North West', 'Yorkshire and The Humber', 'East Midlands', 'West Midlands', 'East of England', 'London', 'South East', 'South West']

── Long format shape  : (348, 3)
── Regions            : ['East Midlands', 'East of England', 'England', 'England and Wales', 'London', 'North East', 'North West', 'South East', 'South West', 'Wales', 'West Midlands Region', 'Yorkshire and The Humber']
── Year range         : 1997 → 2025

── Missing values:
year                   0
affordability_ratio    0
region                 0
dtype: int64


,year,affordability_ratio,region
0,1997,3.17,East Midlands
1,1998,3.26,East Midlands
2,1999,3.42,East Midlands
3,2000,3.46,East Midlands
4,2001,3.72,East Midlands
5,2002,4.18,East Midlands
6,2003,5.04,East Midlands
7,2004,5.94,East Midlands
8,2005,6.07,East Midlands
9,2006,6.15,East Midlands


In [13]:
# ============================================================
# CELL 7 — Load lower quartile affordability ratio (Sheet 2c)
#
# Lower quartile ratio captures affordability for households
# at the bottom 25% of both price and earnings distributions.
# This is more sensitive to affordability stress than the
# median ratio — important for identifying at-risk regions.
# We load it now and merge it into master dataset later.
# ============================================================

ratio_lq_raw = pd.read_excel(
    RAW + 'ons_hp_earnings_ratio.xlsx',
    sheet_name='2c',
    header=1
)

ratio_lq_filtered = ratio_lq_raw[
    ratio_lq_raw['Name'].isin(RATIO_REGION_MAP.keys())
].copy()

year_cols_lq = [c for c in ratio_lq_filtered.columns
                if str(c).strip().isdigit() and 1997 <= int(str(c).strip()) <= 2025]

ratio_lq_long = ratio_lq_filtered.melt(
    id_vars=['Code', 'Name'],
    value_vars=year_cols_lq,
    var_name='year',
    value_name='affordability_ratio_lq'
)

ratio_lq_long['year'] = pd.to_numeric(ratio_lq_long['year'], errors='coerce').astype('Int64')
ratio_lq_long['affordability_ratio_lq'] = pd.to_numeric(ratio_lq_long['affordability_ratio_lq'], errors='coerce')
ratio_lq_long['region'] = ratio_lq_long['Name'].map(RATIO_REGION_MAP)
ratio_lq_long = ratio_lq_long.drop(columns=['Code', 'Name'])
ratio_lq_long = ratio_lq_long.dropna(subset=['affordability_ratio_lq'])
ratio_lq_long = ratio_lq_long.sort_values(['region', 'year']).reset_index(drop=True)

print(f"── LQ ratio shape  : {ratio_lq_long.shape}")
print(f"── Year range      : {ratio_lq_long['year'].min()} → {ratio_lq_long['year'].max()}")
print(f"── Missing values:\n{ratio_lq_long.isnull().sum()}")
ratio_lq_long.head(6)

── LQ ratio shape  : (348, 3)
── Year range      : 1997 → 2025
── Missing values:
year                      0
affordability_ratio_lq    0
region                    0
dtype: int64


,year,affordability_ratio_lq,region
0,1997,3.23,East Midlands
1,1998,3.29,East Midlands
2,1999,3.34,East Midlands
3,2000,3.37,East Midlands
4,2001,3.54,East Midlands
5,2002,4.05,East Midlands


In [14]:
# ── Save to processed ─────────────────────────────────────────
ratio_long.to_csv(PROCESSED + 'ratio_median_clean.csv', index=False)
ratio_lq_long.to_csv(PROCESSED + 'ratio_lq_clean.csv', index=False)

print("✓ ratio_median_clean.csv saved")
print(f"  Shape : {ratio_long.shape}")
print("✓ ratio_lq_clean.csv saved")
print(f"  Shape : {ratio_lq_long.shape}")

✓ ratio_median_clean.csv saved
  Shape : (348, 3)
✓ ratio_lq_clean.csv saved
  Shape : (348, 3)
